# Hälsostudie
**Syftet med den här studien är att undersöka mönster om en persons livstil och om det påverkar deras chanser att få sjukdom.**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from studie_analyzer import HealthAnalyzer
from studie_funktioner import (
    calculate_disease_rate,
    simulate_disease_rate
)

np.random.seed(42)

ModuleNotFoundError: No module named 'studie_analyzer'

In [ ]:
df = pd.read_csv("../data/health_study_dataset.csv")

analyzer = HealthAnalyzer(df)

statistics = analyzer.descriptive_statistics()
display(statistics)

In [ ]:
df.head()
df.info()
df.describe()
df.isnull().sum()

# 1. Beskrivande Analys

In [ ]:
#Räknar ut medelvärde, median, minsta tal, och största tal för dessa kategorier

kolumner = ["age", "weight", "height", "systolic_bp", "cholesterol"]

stats = df[kolumner].agg(["mean", "median", "min", "max"])

stats

In [ ]:
antal_rökare = (
    df["smoker"].value_counts(normalize=True) * 100
)

plt.figure(figsize=(6,4))
plt.hist(df["systolic_bp"], bins=20)
plt.grid(axis="y")
plt.xlabel("Systoliskt blodtryck (i mmHg)")
plt.ylabel("Antal personer")
plt.title("Fördelning av systoliskt blodtryck")
plt.show()

plt.figure(figsize=(6,4))
df.boxplot(column="weight", by="sex")
plt.xlabel("Kön")
plt.ylabel("Vikt (i kg)")
plt.title("Viktfördelning per kön")
plt.suptitle("")
plt.show()

antal_rökare.plot(kind="bar")
plt.grid(axis="y")
plt.xlabel("Rökvanor")
plt.ylabel("Andel (%)")
plt.title("Andel rökare och icke-rökare")
plt.show()

## Tolkning av datan
- Medelvärdet visar den genomsnittliga värdet för respektive variabel,
medan medianen visar det mittersta värdet i datasetet.
- Minimum och maximum visar spridningen mellan det lägsta och högsta värden för respektive variabel.

## Graferna
- Vi ser att majoriteten ligger på ungefär 150 mmHg blodtryck.
- Män har ett större median och medelvärdet än kvinnor när det kommer till vikt, men kvinnor har en större spridning.
- Över 70% av folket i studien röcker inte.

## 2. Simulerad koppling till caset
- I den här delen så undersöker vi förekomsten av sjukdomen i datan. Först räknar vi hur många människor har sjukdomen, sen simulerar vi 1000 personer med samma chans för sjukdomen, sen jämföra.

In [ ]:
# Beräknar hur många personer har sjukdomen i procent

sjukdom_chans = df["disease"].mean()

print(f"Antal människor med sjukdomen: {sjukdom_chans:.2%}")

In [ ]:
# Simulation av 1000 personer

simulation = np.random.binomial(
    n=1,
    p=sjukdom_chans,
    size=1000
)

simulerad_chans = simulation.mean()

print(f"Simulerad mängd av folk med sjukdomen i en sample size på 1000: {simulerad_chans:.2%}")

In [ ]:
# Jämförelse

skillnad = abs(sjukdom_chans - simulerad_chans)

print(f"Verklig andel: {sjukdom_chans:.2%}")
print(f"Simulerad andel: {simulerad_chans:.2%}")
print(f"Skillnad: {skillnad:.2%}")

## Observationer av datan
- Den verkliga chansen för sjukdomen är bara 5.88% i datasetet, vilken är ungefär 47 personer.
- I simulationen med en ökning till 1000 personer istället för 800 har siffran gått ner till 5.60%, och det tillsvarar 56 personer.
- Skillnaden är liten. Men även fast simulationen har ett lägre procent så är mer personer påverkade.

## 3. Konfidensintervall

In [ ]:
bp = df["systolic_bp"].dropna()

mean_bp = bp.mean()
std_bp = bp.std()
n = len(bp)

se = std_bp / np.sqrt(n)

fm = 1.96 * se # fm = felmarginal

lo = mean_bp - fm
hi = mean_bp + fm

print(f"Medelvärde: {mean_bp:.2f} mmHg")
print(f"95% CI: [{lo:.2f} - {hi:.2f}] mmHg")

## Tolknig av resultatet
- Intervallet visar osäkerheten av populationens genomsnittliga systoliska blodtryck.
- Ett litet intervall betyder den är mer litbar, medan en större intervall blir mer osäkert. I det här fallet är intervallet 1,78.

## 4. Hypotesprövning
- H₀ = Rökare har inte högre medel-blodtryck än icke-rökare.
- H₁ = Rökare har högre medel-blodtryck än icke-rökare.
- Signifikansnivå: 0.05, dvs 5% är gränsen för statistisk signifikans.

In [ ]:
rökare = df.loc[df["smoker"] == "Yes", "systolic_bp"].dropna()

icke_rökare = df.loc[df["smoker"] == "No", "systolic_bp"].dropna()

print("Medel-blodtryck för rökare:", rökare.mean())
print("Medel-blodtryck för icke-rökare:", icke_rökare.mean())

In [ ]:
t_statistic, p_value = ttest_ind(
    rökare,
    icke_rökare,
    equal_var=False,
    alternative="greater"
)

print(f"T-statistik: {t_statistic:.4f}")
print(f"P-värde: {p_value:.4f}")

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Resultatet har statistisk signifikans.")
else:
    print("Resultatet har ingen statistisk signifikans.")            

## Tolkning
- Genom ett Welch t-test så undersöker vi om rökare har ett högre medel-blodtryck än dem som inte röker.
- Signifikansnivån sats till 0,05, och P-värdet jämfördes med den för att se om resultatet var statistiskt signifikant.
- P-värdet är mycket högre än 0,05, så det finns ingen statistisk signifikans.